# WiFi CSI People Counting
**Runtime**: Change to **L4 GPU** (Runtime > Change runtime type)

In [ ]:
!pip install -q lightgbm pyarrow

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time, gc, os, json, shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import psutil
print(f'RAM: {psutil.virtual_memory().total/1e9:.1f} GB')

In [ ]:
# Mount Drive and copy file to local (MUCH faster I/O)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Find file in Drive
DRIVE_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train_compact.parquet' in files:
        DRIVE_PATH = os.path.join(root, 'train_compact.parquet')
        break
print(f'Drive: {DRIVE_PATH}')

# Copy to local SSD for fast reads
LOCAL_PATH = '/content/train_compact.parquet'
if not os.path.exists(LOCAL_PATH):
    print('Copying to local storage...')
    shutil.copy(DRIVE_PATH, LOCAL_PATH)
print(f'Local: {os.path.getsize(LOCAL_PATH)/1e9:.2f} GB')

In [ ]:
# Load data
amp_cols = [f'amp_{i}' for i in range(256)]
phase_cols = [f'phase_{i}' for i in range(256)]
table = pq.read_table(LOCAL_PATH, columns=amp_cols + phase_cols + ['num_people'])

y = table.column('num_people').to_numpy().astype(np.int64)
X = np.empty((len(y), 512), dtype=np.float32)
for i, col in enumerate(amp_cols + phase_cols):
    X[:, i] = table.column(col).to_numpy().astype(np.float32)
del table; gc.collect()

print(f'X: {X.shape} ({X.nbytes/1e9:.2f} GB), Classes: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Explore
dist = pd.Series(y).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dist.plot(kind='bar', ax=axes[0], color=plt.cm.viridis(np.linspace(0.2, 0.8, 6)))
axes[0].set_title('People Count Distribution'); axes[0].set_xlabel('People'); axes[0].set_ylabel('Count')
for n in sorted(dist.index):
    axes[1].plot(X[np.where(y==n)[0][0], :256], label=f'{n} people', alpha=0.7)
axes[1].set_title('Sample CSI Amplitude'); axes[1].set_xlabel('Subcarrier'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Normalize in-place
amp_mean, amp_std = X[:,:256].mean(0), X[:,:256].std(0) + 1e-8
phase_mean, phase_std = X[:,256:].mean(0), X[:,256:].std(0) + 1e-8
X[:,:256] = (X[:,:256] - amp_mean) / amp_std
X[:,256:] = (X[:,256:] - phase_mean) / phase_std

# Split by indices
idx = np.arange(len(y))
idx_train, idx_temp = train_test_split(idx, test_size=0.3, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42, stratify=y[idx_temp])
del idx_temp

y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
class_counts = np.bincount(y_train, minlength=6)
class_weights = len(y_train) / (6 * class_counts + 1)

print(f'Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')

## LightGBM (GPU-accelerated)

In [ ]:
%%time
sample_weights = np.array([class_weights[c] for c in y_train])
lgb_train = lgb.Dataset(X[idx_train], label=y_train, weight=sample_weights)
lgb_val = lgb.Dataset(X[idx_val], label=y_val, reference=lgb_train)

params = {
    'objective': 'multiclass', 'num_class': 6, 'metric': 'multi_logloss',
    'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 50,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'device': 'gpu', 'gpu_use_dp': False,
    'verbose': -1, 'n_jobs': -1, 'seed': 42,
}

lgb_model = lgb.train(
    params, lgb_train, num_boost_round=500,
    valid_sets=[lgb_val], valid_names=['val'],
    callbacks=[lgb.log_evaluation(50), lgb.early_stopping(30)],
)
print(f'Best iteration: {lgb_model.best_iteration}')

In [ ]:
y_pred_lgb = lgb_model.predict(X[idx_test]).argmax(axis=1)
print(f'=== LightGBM Test Accuracy: {accuracy_score(y_test, y_pred_lgb):.4f} ===')
print(classification_report(y_test, y_pred_lgb, target_names=[f'{i} people' for i in range(6)]))

cm = confusion_matrix(y_test, y_pred_lgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'{i}p' for i in range(6)], yticklabels=[f'{i}p' for i in range(6)])
plt.title(f'LightGBM (Acc: {accuracy_score(y_test, y_pred_lgb):.3f})')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()

## 1D-CNN (GPU)

In [ ]:
class CSI_CNN(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = CSI_CNN().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Build dataloaders with large batch size for L4
def make_dataset(indices):
    d = X[indices]
    x_2ch = np.stack([d[:,:256], d[:,256:]], axis=1)
    return TensorDataset(torch.tensor(x_2ch), torch.tensor(y[indices]))

train_ds = make_dataset(idx_train)
val_ds = make_dataset(idx_val)
test_ds = make_dataset(idx_test)
del X, y; gc.collect()

BS = 4096  # Large batch for L4
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
# Train with mixed precision for speed
weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_t)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
scaler = torch.amp.GradScaler()

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
SAVE_PATH = '/content/drive/MyDrive/fyp_models'
os.makedirs(SAVE_PATH, exist_ok=True)

for epoch in range(30):
    t0 = time.time()
    model.train()
    tloss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tloss += loss.item() * len(xb)
    tloss /= len(train_loader.dataset)

    model.eval()
    vloss, correct = 0, 0
    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            vloss += criterion(out, yb).item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()
    vloss /= len(val_loader.dataset)
    vacc = correct / len(val_loader.dataset)
    scheduler.step()

    history['train_loss'].append(tloss)
    history['val_loss'].append(vloss)
    history['val_acc'].append(vacc)

    tag = ''
    if vacc > best_val_acc:
        best_val_acc = vacc
        torch.save(model.state_dict(), f'{SAVE_PATH}/best_cnn_model.pt')
        tag = ' *best*'

    print(f'Epoch {epoch+1:2d}/30 | TL: {tloss:.4f} | VL: {vloss:.4f} | VA: {vacc:.4f} | {time.time()-t0:.1f}s{tag}')

print(f'\nBest: {best_val_acc:.4f}')

In [ ]:
# Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3); ax1.set_title('Loss')
ax2.plot(history['val_acc'], color='green')
ax2.axhline(best_val_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3); ax2.set_title('Val Accuracy')
plt.tight_layout(); plt.show()

In [ ]:
# CNN test evaluation
model.load_state_dict(torch.load(f'{SAVE_PATH}/best_cnn_model.pt', weights_only=True))
model.eval()
preds, labels = [], []
with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
    for xb, yb in test_loader:
        preds.append(model(xb.to(device)).argmax(1).cpu().numpy())
        labels.append(yb.numpy())
y_pred_cnn = np.concatenate(preds)
y_true = np.concatenate(labels)

print(f'=== CNN Test Accuracy: {accuracy_score(y_true, y_pred_cnn):.4f} ===')
print(classification_report(y_true, y_pred_cnn, target_names=[f'{i} people' for i in range(6)]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm_cnn = confusion_matrix(y_true, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[f'{i}p' for i in range(6)], yticklabels=[f'{i}p' for i in range(6)])
axes[0].set_title(f'CNN (Acc: {accuracy_score(y_true, y_pred_cnn):.3f})')
cm_norm = cm_cnn.astype(float) / cm_cnn.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=[f'{i}p' for i in range(6)], yticklabels=[f'{i}p' for i in range(6)])
axes[1].set_title('Normalized')
plt.tight_layout(); plt.show()

In [ ]:
# Save all results to Drive
lgb_model.save_model(f'{SAVE_PATH}/lgb_people_counting.txt')
np.savez(f'{SAVE_PATH}/norm_stats.npz', amp_mean=amp_mean, amp_std=amp_std, phase_mean=phase_mean, phase_std=phase_std)

f1_lgb = f1_score(y_test, y_pred_lgb, average=None)
f1_cnn = f1_score(y_true, y_pred_cnn, average=None)

results = {
    'lgb_accuracy': float(accuracy_score(y_test, y_pred_lgb)),
    'cnn_accuracy': float(accuracy_score(y_true, y_pred_cnn)),
    'lgb_f1_per_class': {str(i): round(float(f),4) for i,f in enumerate(f1_lgb)},
    'cnn_f1_per_class': {str(i): round(float(f),4) for i,f in enumerate(f1_cnn)},
    'history': history,
}
with open(f'{SAVE_PATH}/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {SAVE_PATH}:')
for f in os.listdir(SAVE_PATH): print(f'  {f}')

## Summary - Copy this output and send it back!

In [ ]:
print('=== COPY EVERYTHING BELOW ===')
print(json.dumps(results, indent=2))
print('\n=== LightGBM ===')
print(classification_report(y_test, y_pred_lgb, target_names=[f'{i}p' for i in range(6)]))
print('=== CNN ===')
print(classification_report(y_true, y_pred_cnn, target_names=[f'{i}p' for i in range(6)]))